# GCI World 2026 - Single-Stream Champion (v17 only, one CSV)

This notebook is the v17 stream of the V22 champion, **without** the second stream (v18) and **without** the rank-fusion of two CSVs.
Your own records show v17 = 0.78237, v18 = 0.78235 and the fusion = 0.78239, so the fusion adds +0.00002 (far below public-leaderboard noise). One stream gives one CSV.

What changed versus the notebook you pasted:
1. **Cache bug fixed.** The old `_MEMO` key `(len(fit), len(qi), fit[0])` collides across folds, so encodings were silently built for the wrong rows. Keys are now content hashes.
2. **Stream 2 and the fusion cell removed.** Only `v_single_stream_v17.csv` is written.
3. **No pre-written outputs.** Every number is printed by the code that computes it. There is no "Matches Public LB" text.
4. **`STRICT_INDUCTIVE` switch** (default `False` = champion-faithful). Set `GCI_STRICT=1` (or edit the flag) to compute every statistic from `train.csv` rows only (counts, group means, partner links, residual regressors, fee flag), so no test-row covariate enters feature construction.
5. Safety checks: test ID order, NaN/row-count checks, optional parity check against a reference CSV.

After running: compare the printed OOF AUC with 0.7765 and run the parity cell against your saved v17 CSV (Spearman should be about 0.999 or higher).

In [ ]:
# Cell 1: Environment
import subprocess, sys, os, glob, time, hashlib, warnings
warnings.filterwarnings('ignore')
for pkg, mod in (('lightgbm', 'lightgbm'), ('catboost', 'catboost'), ('scikit-learn', 'sklearn'), ('scipy', 'scipy'), ('pandas', 'pandas'), ('numpy', 'numpy')):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', pkg])

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingRegressor as HR, HistGradientBoostingClassifier as HC

# False = champion-faithful (statistics pooled over train+test rows). True = every statistic comes from train rows only.
STRICT_INDUCTIVE = os.environ.get('GCI_STRICT', '0') == '1'
print('LightGBM', lgb.__version__, '| CPU threads', os.cpu_count(), '| STRICT_INDUCTIVE =', STRICT_INDUCTIVE)

In [ ]:
# Cell 2: Data discovery
def find_data():
    override = os.environ.get('GCI_DATA_DIR')
    if override:
        return [os.path.join(override, x) for x in ('train.csv', 'test.csv', 'sample_submission.csv')]
    candidates = []
    roots = ('/content', '/content/drive/MyDrive', 'd:/gci/inpuy', 'd:/gci/input', os.getcwd())
    for r in roots:
        if os.path.isdir(r):
            for pat in ('train.csv', '*/train.csv', '*/*/train.csv'):
                candidates += glob.glob(os.path.join(r, pat))
    for f in sorted(set(candidates)):
        d = os.path.dirname(f)
        if all(os.path.exists(os.path.join(d, x)) for x in ('test.csv', 'sample_submission.csv')):
            try:
                head = open(f, 'rb').readline().decode('utf-8', 'ignore').strip().split(',')
                nrows = sum(1 for _ in open(f, 'rb')) - 1
            except Exception:
                continue
            if len(head) == 34 and 'TARGET' in head and nrows == 171202:
                return [os.path.join(d, x) for x in ('train.csv', 'test.csv', 'sample_submission.csv')]
    raise FileNotFoundError('train.csv, test.csv, sample_submission.csv not found')

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = find_data()
df_train = pd.read_csv(TRAIN_PATH); df_test = pd.read_csv(TEST_PATH); sample = pd.read_csv(SAMPLE_PATH)
y = df_train['TARGET'].values.astype(np.int32)
ntr, nte = len(df_train), len(df_test)
tidx = np.arange(ntr, ntr + nte)
assert (sample['SK_ID_CURR'].values == df_test['SK_ID_CURR'].values).all(), 'sample_submission order differs from test.csv'
print(f'data: {os.path.dirname(TRAIN_PATH)} | train {df_train.shape} test {df_test.shape} | target rate {y.mean():.4f}')

In [ ]:
# Cell 3: Feature engineering (champion definitions; pooled vs train-only statistics controlled by STRICT_INDUCTIVE)
CATS = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

al = pd.concat([df_train.drop(columns=['TARGET']), df_test], ignore_index=True)
REF = al.iloc[:ntr] if STRICT_INDUCTIVE else al      # rows allowed to contribute statistics

def new_cols(d):
    return pd.DataFrame(d, index=al.index)

def base(al):
    X = al.drop(columns=['SK_ID_CURR']).copy()
    for c in CATS:
        X[c] = X[c].astype('category').cat.codes.replace(-1, np.nan).astype(float)
    return X

def f_clean(al, X):
    d = {'DE_SENT': (al.DAYS_EMPLOYED == 365243).astype(int),
         'DAYS_EMPLOYED_CLEAN': al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243),
         'ORG_XNA': (al.ORGANIZATION_TYPE == 'XNA').astype(int),
         'CAR_AGE_OUT': (al.OWN_CAR_AGE >= 60).astype(int),
         'MISS_CNT': al.isna().sum(axis=1)}
    for c in ['EXT_SOURCE_1', 'EXT_SOURCE_3', 'OWN_CAR_AGE', 'AMT_REQ_CREDIT_BUREAU_YEAR']:
        d[c + '_NA'] = al[c].isna().astype(int)
    return pd.concat([X, new_cols(d)], axis=1)

def f_ext(al, X):
    e = al[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']]
    d = {'EXT_MEAN': e.mean(axis=1), 'EXT_MIN': e.min(axis=1), 'EXT_MAX': e.max(axis=1), 'EXT_STD': e.std(axis=1),
         'EXT_RANGE': e.max(axis=1) - e.min(axis=1), 'EXT_NOBS': e.notna().sum(axis=1), 'EXT_PROD': e.prod(axis=1),
         'EXT_W': (al.EXT_SOURCE_1 * 2 + al.EXT_SOURCE_2 * 3 + al.EXT_SOURCE_3 * 4) / 9.0,
         'E12': al.EXT_SOURCE_1 * al.EXT_SOURCE_2, 'E13': al.EXT_SOURCE_1 * al.EXT_SOURCE_3, 'E23': al.EXT_SOURCE_2 * al.EXT_SOURCE_3,
         'D12': al.EXT_SOURCE_1 - al.EXT_SOURCE_2, 'D13': al.EXT_SOURCE_1 - al.EXT_SOURCE_3, 'D23': al.EXT_SOURCE_2 - al.EXT_SOURCE_3}
    return pd.concat([X, new_cols(d)], axis=1)

def f_fin(al, X):
    inc, cr, an, gp = al.AMT_INCOME_TOTAL, al.AMT_CREDIT, al.AMT_ANNUITY, al.AMT_GOODS_PRICE
    d = {'CREDIT_INC': cr / inc, 'ANN_INC': an / inc, 'ANN_CREDIT': an / cr, 'TERM': cr / an,
         'GOODS_CREDIT': gp / cr, 'CREDIT_MINUS_GOODS': cr - gp, 'GOODS_INC': gp / inc,
         'INC_PER_FAM': inc / al.CNT_FAM_MEMBERS, 'CREDIT_PER_FAM': cr / al.CNT_FAM_MEMBERS, 'ANN_PER_FAM': an / al.CNT_FAM_MEMBERS}
    for c in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']:
        d['LOG_' + c] = np.log1p(al[c])
    return pd.concat([X, new_cols(d)], axis=1)

def f_age(al, X):
    age = -al.DAYS_BIRTH
    de = al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243)
    d = {'AGE': age / 365.25, 'EMP_AGE': de / al.DAYS_BIRTH, 'EMP_YEARS': -de / 365.25,
         'REG_AGE': al.DAYS_REGISTRATION / al.DAYS_BIRTH, 'ID_AGE': al.DAYS_ID_PUBLISH / al.DAYS_BIRTH,
         'PHONE_AGE': al.DAYS_LAST_PHONE_CHANGE / al.DAYS_BIRTH, 'CAR_AGE_RATIO': al.OWN_CAR_AGE / (age / 365.25),
         'CREDIT_PER_AGE': al.AMT_CREDIT / (age / 365.25), 'INC_PER_AGE': al.AMT_INCOME_TOTAL / (age / 365.25)}
    return pd.concat([X, new_cols(d)], axis=1)

def f_bureau(al, X):
    b = [c for c in al.columns if c.startswith('AMT_REQ')]
    d = {'BUR_SUM': al[b].sum(axis=1, min_count=1),
         'BUR_RECENT': al[['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON']].sum(axis=1, min_count=1),
         'BUR_MON_YEAR': al.AMT_REQ_CREDIT_BUREAU_MON / (al.AMT_REQ_CREDIT_BUREAU_YEAR + 1.0)}
    return pd.concat([X, new_cols(d)], axis=1)

def f_count(al, X):
    d = {}
    for c in CATS + ['REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_ANNUITY', 'AMT_INCOME_TOTAL']:
        m = al[c].map(REF[c].value_counts())
        if STRICT_INDUCTIVE:
            m = m.where(m.notna() | al[c].isna(), 0.0)      # value never seen in train -> count 0
        d['CNT_' + c] = m
    return pd.concat([X, new_cols(d)], axis=1)

def f_group(al, X):
    tmp = al.copy(); tmp['CREDIT_INC'] = al.AMT_CREDIT / al.AMT_INCOME_TOTAL
    rt = tmp.loc[REF.index]
    d = {}
    for k in ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'REGION_POPULATION_RELATIVE']:
        for v in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH', 'CREDIT_INC']:
            d[f'G_{k[:6]}_{v[-6:]}_DIFF'] = tmp[v] - tmp[k].map(rt.groupby(k)[v].mean())
    return pd.concat([X, new_cols(d)], axis=1)

def f_kdec(al, X):
    C, G = al.AMT_CREDIT, al.AMT_GOODS_PRICE
    cash = al.NAME_CONTRACT_TYPE == 'Cash loans'
    kf = (C / G - 1.0) / 0.0132
    k = kf.round()
    ok = cash & (C > G) & ((kf - k).abs() < 0.1)
    K = pd.Series(np.where(ok, k, np.where(cash & (C == G), 0.0, np.nan)), index=al.index)
    return pd.concat([X, new_cols({'K_DEC': K, 'CG_RATIO': C / G})], axis=1)

# Repeat-applicant partner linkage (members are limited to train rows when STRICT_INDUCTIVE)
_i1 = al.DAYS_REGISTRATION - al.DAYS_BIRTH
_i2 = al.DAYS_ID_PUBLISH - al.DAYS_BIRTH
_gb = al.DAYS_BIRTH.values

def _groups(parts):
    k = al[parts].astype(str).agg('|'.join, axis=1) + '|' + _i1.astype(str) + '|' + _i2.astype(str)
    k = k.where(al.DAYS_REGISTRATION != 0)
    sz = k.map(k.value_counts())
    k = k.where(sz > 1)
    g = pd.Series(pd.factorize(k)[0]).replace(-1, np.nan).values
    return pd.DataFrame({'g': g}).groupby('g').groups

def _plist(mem):
    out = {}
    for _, ix in mem.items():
        ix = np.asarray(ix)
        pool = ix[ix < ntr] if STRICT_INDUCTIVE else ix
        for i in ix:
            o = pool[pool != i]
            if len(o) == 0:
                continue
            out[i] = (o, _gb[o] - _gb[i])
    return out

PL_S = _plist(_groups(['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'REGION_POPULATION_RELATIVE', 'FLAG_MOBIL']))
PL_L = _plist(_groups(['CODE_GENDER']))

def f_partner(al, X):
    N = len(al)
    gap = np.full(N, np.nan); has = np.zeros(N); pn = np.full(N, np.nan); hl = np.zeros(N); pnl = np.full(N, np.nan)
    for i, (o, gp_) in PL_S.items():
        gap[i] = gp_[np.argmin(np.abs(gp_))]; has[i] = 1; pn[i] = len(o) + 1
    for i, (o, _) in PL_L.items():
        hl[i] = 1; pnl[i] = len(o) + 1
    d = {'HAS_PARTNER': has.astype(int), 'PARTNER_N': pn, 'PARTNER_GAP': gap, 'PARTNER_ABSGAP': np.abs(gap),
         'HAS_PARTNER_LOOSE': hl.astype(int), 'PARTNER_N_LOOSE': pnl}
    return pd.concat([X, new_cols(d)], axis=1)

def f_person(al, X):
    N = len(al)
    E = al[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].values.astype(float)
    pe = np.full((N, 3), np.nan); dif = np.full((N, 3), np.nan)
    cr = np.full(N, np.nan); an = np.full(N, np.nan); inc = np.full(N, np.nan); same = np.full(N, np.nan); qd = np.full(N, np.nan)
    Qv = (al.AMT_ANNUITY / al.AMT_CREDIT).values
    ct = (al.NAME_CONTRACT_TYPE == 'Cash loans').values
    CR, AN_, INC = al.AMT_CREDIT.values, al.AMT_ANNUITY.values, al.AMT_INCOME_TOTAL.values
    for i, (o, g) in PL_S.items():
        j = o[np.argmin(np.abs(g))]
        allr = np.r_[i, o]
        for k in range(3):
            v = E[allr, k]
            pe[i, k] = np.nanmean(v) if np.isfinite(v).any() else np.nan
            dif[i, k] = E[i, k] - E[j, k]
        cr[i] = CR[i] / CR[j]; an[i] = AN_[i] / AN_[j]; inc[i] = INC[i] / INC[j]
        same[i] = float(ct[i] == ct[j]); qd[i] = Qv[i] - Qv[j]
    d = {f'PERS_EXT{k+1}': pe[:, k] for k in range(3)}
    d.update({f'PDIF_EXT{k+1}': dif[:, k] for k in range(3)})
    d.update(PCR_RATIO=cr, PAN_RATIO=an, PINC_RATIO=inc, PSAME_CT=same, PQ_DIFF=qd)
    P = pd.DataFrame(d)
    P['PERS_EXT_MEAN'] = P[['PERS_EXT1', 'PERS_EXT2', 'PERS_EXT3']].mean(axis=1)
    return pd.concat([X, new_cols({c: P[c].values for c in P.columns})], axis=1)

X_b = base(al)
Xc = f_clean(al, X_b); Xc = f_fin(al, Xc); Xc = f_age(al, Xc); Xc = f_bureau(al, Xc); Xc = f_partner(al, Xc); Xc = f_person(al, Xc)
Xa = f_ext(al, Xc); Xa = f_count(al, Xa); Xa = f_group(al, Xa)
Xck = f_kdec(al, Xc); Xak = f_kdec(al, Xa)

# Decoded product keys (fee flag counts come from REF rows)
q = al.AMT_ANNUITY / al.AMT_CREDIT
_cash = (al.NAME_CONTRACT_TYPE == 'Cash loans').values
_qb = (al.AMT_ANNUITY - 129.5) / al.AMT_CREDIT
_q5, _qb5 = q.round(5), _qb.round(5)
def _refcount(s):
    return s.map(s.loc[REF.index].value_counts()).fillna(0).values
FEE = ((_refcount(_qb5) > _refcount(_q5)) & _cash).astype(int)
Q0 = pd.Series(np.where(FEE, _qb, q))

KS = {'Q3': q.round(3).astype(str), 'Q4': q.round(4).astype(str), 'Q5': q.round(5).astype(str),
      'P3': Q0.round(3).astype(str), 'P4': Q0.round(4).astype(str), 'P5': Q0.round(5).astype(str),
      'PF4': Q0.round(4).astype(str) + '|' + pd.Series(FEE).astype(str)}
M0 = {'Q3': 100., 'Q4': 30., 'Q5': 30., 'P3': 100., 'P4': 30., 'P5': 30., 'PF4': 30.}
CT = al.AMT_CREDIT / al.AMT_ANNUITY
CT1, CT2, CT3 = CT.round(1).astype(str), CT.round(2).astype(str), CT.round(3).astype(str)
KS.update({'CT1': CT1, 'CT2': CT2, 'CT3': CT3, 'CT1_s10': CT1, 'CT1_s200': CT1,
           'CT1_GP': CT1 + '|' + al.AMT_GOODS_PRICE.astype(str), 'CT1_CR': CT1 + '|' + al.AMT_CREDIT.astype(str), 'CT1_AN': CT1 + '|' + al.AMT_ANNUITY.astype(str),
           'GP': al.AMT_GOODS_PRICE.astype(str), 'CR': al.AMT_CREDIT.astype(str), 'AN': al.AMT_ANNUITY.astype(str)})
M0.update({'CT1': 50., 'CT2': 50., 'CT3': 50., 'CT1_s10': 10., 'CT1_s200': 200., 'CT1_GP': 50., 'CT1_CR': 50., 'CT1_AN': 50., 'GP': 50., 'CR': 50., 'AN': 50.})

BASEF = {'Xc': Xc.iloc[:ntr].reset_index(drop=True), 'Xct': Xc.iloc[ntr:].reset_index(drop=True),
         'Xa': Xa.iloc[:ntr].reset_index(drop=True), 'Xat': Xa.iloc[ntr:].reset_index(drop=True),
         'Xck': Xck.iloc[:ntr].reset_index(drop=True), 'Xckt': Xck.iloc[ntr:].reset_index(drop=True),
         'Xak': Xak.iloc[:ntr].reset_index(drop=True), 'Xakt': Xak.iloc[ntr:].reset_index(drop=True)}
print(f'features: Core={BASEF["Xc"].shape[1]} All={BASEF["Xa"].shape[1]} | partner rows strict={len(PL_S)} loose={len(PL_L)} | fee-flag share (cash)={FEE[_cash].mean():.3f} | test rows with partner={sum(1 for i in PL_S if i >= ntr)}')

In [ ]:
# Cell 4: 16 demographic learned-expectation residuals
raw = al.drop(columns=['SK_ID_CURR']).copy()
for c in CATS:
    raw[c] = raw[c].astype('category').cat.codes.replace(-1, np.nan).astype(float)
raw['DAYS_EMPLOYED'] = raw.DAYS_EMPLOYED.where(raw.DAYS_EMPLOYED != 365243)
demo = [c for c in raw.columns if c.startswith('EXT_SOURCE') or c.startswith('AMT_')]
T = {'EXT1': (raw.EXT_SOURCE_1, ['EXT_SOURCE_1']), 'EXT2': (raw.EXT_SOURCE_2, ['EXT_SOURCE_2']), 'EXT3': (raw.EXT_SOURCE_3, ['EXT_SOURCE_3']),
     'LOGINC': (np.log(raw.AMT_INCOME_TOTAL), ['AMT_INCOME_TOTAL']), 'LOGCR': (np.log(raw.AMT_CREDIT), ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']),
     'EMP': (raw.DAYS_EMPLOYED, ['DAYS_EMPLOYED']), 'REG': (raw.DAYS_REGISTRATION, ['DAYS_REGISTRATION']), 'AGE': (raw.DAYS_BIRTH, ['DAYS_BIRTH']),
     'IDP': (raw.DAYS_ID_PUBLISH, ['DAYS_ID_PUBLISH']), 'PHONE': (raw.DAYS_LAST_PHONE_CHANGE, ['DAYS_LAST_PHONE_CHANGE']),
     'REGPOP': (raw.REGION_POPULATION_RELATIVE, ['REGION_POPULATION_RELATIVE']), 'FAM': (raw.CNT_FAM_MEMBERS, ['CNT_FAM_MEMBERS']), 'CAR': (raw.OWN_CAR_AGE, ['OWN_CAR_AGE']),
     'EXT1D': (raw.EXT_SOURCE_1, demo), 'EXT2D': (raw.EXT_SOURCE_2, demo), 'EXT3D': (raw.EXT_SOURCE_3, demo)}

mkreg = lambda: HR(max_iter=100, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=100, random_state=0)
res_out = {}
for nm, (yt, drop_cols) in T.items():
    Xp = raw.drop(columns=drop_cols)
    pred = np.zeros(len(raw)); obs = yt.notna().values
    if not STRICT_INDUCTIVE:
        for tri, tei in KFold(5, shuffle=True, random_state=0).split(Xp):
            m = mkreg().fit(Xp.iloc[tri][obs[tri]], yt.iloc[tri][obs[tri]])
            pred[tei] = m.predict(Xp.iloc[tei])
    else:                                              # fit on train rows only; test rows get the fold-model average
        tp = np.zeros(nte)
        for tri, tei in KFold(5, shuffle=True, random_state=0).split(np.arange(ntr)):
            m = mkreg().fit(Xp.iloc[tri][obs[tri]], yt.iloc[tri][obs[tri]])
            pred[tei] = m.predict(Xp.iloc[tei]); tp += m.predict(Xp.iloc[ntr:]) / 5.0
        pred[ntr:] = tp
    res_out[f'RP_{nm}'] = pred
    res_out[f'RR_{nm}'] = np.where(obs, yt.values - pred, np.nan)

R_df = pd.DataFrame(res_out)
R_tr, R_te = R_df.iloc[:ntr].reset_index(drop=True), R_df.iloc[ntr:].reset_index(drop=True)
F1 = {k: pd.concat([BASEF[k], R_te if k.endswith('t') else R_tr], axis=1) for k in BASEF}
print(f'residuals done: Core={F1["Xc"].shape[1]} All={F1["Xa"].shape[1]}')

In [ ]:
# Cell 5: multi-view stacking features (v17 views: EXT, LOAN, PERSON)
lg = lambda p: np.log(np.clip(p, 1e-4, 1.0 - 1e-4) / (1.0 - np.clip(p, 1e-4, 1.0 - 1e-4)))

def compute_views(variant):
    Xs = F1['Xa'].iloc[:ntr].copy(); Xts = F1['Xat'].copy()
    Xs['Q0num'] = Q0.values[:ntr]; Xts['Q0num'] = Q0.values[ntr:]
    cols = list(Xs.columns)
    ext = [c for c in cols if 'EXT' in c or c in ('E12', 'E13', 'E23', 'D12', 'D13', 'D23')]
    loan = [c for c in cols if c.startswith(('AMT_', 'LOG_', 'CREDIT_', 'ANN_', 'GOODS_', 'TERM', 'Q0num')) or c in ('CREDIT_MINUS_GOODS', 'RP_LOGINC', 'RR_LOGINC', 'RP_LOGCR', 'RR_LOGCR')]
    person = [c for c in cols if c not in set(ext) | set(loan) and not c.startswith(('G_', 'CNT_'))]
    views = {'EXT': ext, 'LOAN': loan, 'PERSON': person}

    def vf(d):
        df = pd.DataFrame({f'VIEW_{k}': v for k, v in d.items()})
        df['VD_EL'] = df.VIEW_EXT - df.VIEW_LOAN; df['VD_EP'] = df.VIEW_EXT - df.VIEW_PERSON; df['VD_LP'] = df.VIEW_LOAN - df.VIEW_PERSON
        return df

    mk = lambda: HC(max_iter=100, learning_rate=0.08, max_leaf_nodes=15, min_samples_leaf=150, l2_regularization=20, random_state=0)
    out = {'tr': [], 'va': [], 'te': []}
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(ntr), y)):
        trv, vav, tev = {}, {}, {}
        for vn, vc in views.items():
            inner = np.zeros(len(a)); pvb, pte = [], []
            for ia, ib in StratifiedKFold(3, shuffle=True, random_state=11).split(a, y[a]):
                m = mk().fit(Xs.iloc[a[ia]][vc], y[a[ia]])
                inner[ib] = m.predict_proba(Xs.iloc[a[ib]][vc])[:, 1]
                if variant == 'A':
                    pvb.append(m.predict_proba(Xs.iloc[b][vc])[:, 1]); pte.append(m.predict_proba(Xts[vc])[:, 1])
            trv[vn] = lg(inner)
            if variant == 'A':
                vav[vn] = lg(np.mean(pvb, 0)); tev[vn] = lg(np.mean(pte, 0))
            else:
                m = mk().fit(Xs.iloc[a][vc], y[a])
                vav[vn] = lg(m.predict_proba(Xs.iloc[b][vc])[:, 1]); tev[vn] = lg(m.predict_proba(Xts[vc])[:, 1])
        out['tr'].append(vf(trv)); out['va'].append(vf(vav)); out['te'].append(vf(tev))
    trv, tev = {}, {}
    for vn, vc in views.items():
        inner = np.zeros(ntr); pte = []
        for ia, ib in StratifiedKFold(3, shuffle=True, random_state=11).split(np.zeros(ntr), y):
            m = mk().fit(Xs.iloc[ia][vc], y[ia]); inner[ib] = m.predict_proba(Xs.iloc[ib][vc])[:, 1]
            if variant == 'A':
                pte.append(m.predict_proba(Xts[vc])[:, 1])
        trv[vn] = lg(inner)
        tev[vn] = lg(np.mean(pte, 0)) if variant == 'A' else lg(mk().fit(Xs[vc], y).predict_proba(Xts[vc])[:, 1])
    out['all_tr'] = vf(trv); out['all_te'] = vf(tev)
    print(f'  views ({variant}): sizes', {k: len(v) for k, v in views.items()})
    return out

VIEW_O = compute_views('orig')
VIEW_A = compute_views('A')

In [ ]:
# Cell 6: in-fold encoders (target encodings + partner-label features) with a CORRECT cache
KC = {k: pd.factorize(v)[0] for k, v in KS.items()}
KN = {k: int(v.max()) + 1 for k, v in KC.items()}

def _h(a):
    return hashlib.md5(np.ascontiguousarray(np.asarray(a, dtype=np.int64)).tobytes()).hexdigest()

def partner_enc(fit, idx):
    idx = np.asarray(idx)
    cols = ['PY_ANY', 'PY_N', 'PY_NEAR180', 'PY_EARLIER', 'PY_LATER', 'PYL_ANY']
    out = np.full((len(idx), 6), np.nan); out[:, 1] = 0
    lab = np.full(ntr + nte, np.nan); lab[fit] = y[fit]
    for j, i in enumerate(idx):
        if i in PL_S:
            o, gp_ = PL_S[i]; v = lab[o]; k = ~np.isnan(v)
            if k.any():
                v, gp_ = v[k], gp_[k]; out[j, 0] = v.max(); out[j, 1] = k.sum()
                if (np.abs(gp_) <= 180).any(): out[j, 2] = v[np.abs(gp_) <= 180].max()
                if (gp_ > 0).any(): out[j, 3] = v[gp_ > 0].max()
                if (gp_ < 0).any(): out[j, 4] = v[gp_ < 0].max()
        if i in PL_L:
            o, gp_ = PL_L[i]; v = lab[o]; v = v[~np.isnan(v)]
            if len(v): out[j, 5] = v.max()
    return pd.DataFrame(out, columns=cols)

_MEMO = {}
def enc(fit, qi):
    fit = np.asarray(fit); qi = np.asarray(qi)
    mk = ('enc', _h(fit), _h(qi))
    if mk in _MEMO: return _MEMO[mk]
    pr = float(y[fit].mean()); yf = y[fit].astype(float); out = {}
    for k in KS:
        c = KC[k]; cnt_t = np.bincount(c[fit], minlength=KN[k]).astype(float)
        sm_t = np.bincount(c[fit], weights=yf, minlength=KN[k]); cq = c[qi]
        out[f'TE_{k}'] = (sm_t[cq] + M0[k] * pr) / (cnt_t[cq] + M0[k])
        out[f'CNT_{k}'] = cnt_t[cq]
    res = pd.concat([pd.DataFrame(out), partner_enc(fit, qi)], axis=1)
    _MEMO[mk] = res
    return res

def train_enc(a):
    a = np.asarray(a)
    mk = ('tr', _h(a))
    if mk in _MEMO: return _MEMO[mk]
    cols = enc(a[:10], a[:5]).columns
    inner = np.zeros((len(a), len(cols)))
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=7).split(a, y[a]):
        inner[ib] = enc(a[ia], a[ib]).values
    res = pd.DataFrame(inner, columns=cols)
    _MEMO[mk] = res
    return res

def onehot(Xa_, Xt_):
    z = pd.concat([Xa_, Xt_], ignore_index=True)
    oh = pd.get_dummies(z[CATS].astype('category'), dummy_na=True, dtype=np.float32)
    oh.columns = [f'OH_{i}' for i in range(oh.shape[1])]
    Z = pd.concat([z.drop(columns=CATS), oh], axis=1)
    return Z.iloc[:len(Xa_)].reset_index(drop=True), Z.iloc[len(Xa_):].reset_index(drop=True)

Xc_, Xct_, Xa_, Xat_, Xck_, Xckt_, Xak_, Xakt_ = [F1[k] for k in ('Xc', 'Xct', 'Xa', 'Xat', 'Xck', 'Xckt', 'Xak', 'Xakt')]
Xco, Xcot = onehot(Xc_, Xct_); Xao, Xaot = onehot(Xa_, Xat_); Xaok, Xaokt = onehot(Xak_, Xakt_)

def get_specs():
    MONO_K = [-1 if c in ('EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3') else 0 for c in Xck_.columns] + [0] * (2 * len(KS) + 6)
    P = dict(feature_pre_filter=False)
    SPEC1 = {
        'qf_shallow':   (Xco,  Xcot,  dict(learning_rate=0.05, num_leaves=7, max_depth=3, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20, **P), 1300, 1430, [42, 7, 2024, 11, 99]),
        'qf_reg':       (Xco,  Xcot,  dict(learning_rate=0.03, num_leaves=11, min_child_samples=300, feature_fraction=0.4, bagging_fraction=0.8, lambda_l2=40, **P), 1939, 2133, [42, 7, 2024, 11, 99]),
        'v1_allfam_oh': (Xao,  Xaot,  dict(learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20, **P), 881, 969, [42, 7, 2024]),
        'v2_allfam_oh': (Xaok, Xaokt, dict(learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20, **P), 881, 969, [42, 7, 2024]),
        'v2_mono':      (Xck_.reset_index(drop=True), Xckt_.reset_index(drop=True), dict(learning_rate=0.02, num_leaves=15, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20, monotone_constraints=MONO_K, **P), 1571, 1729, [42, 7, 2024])}
    SPEC1_ET = {nm: (X, Xt, {**p, 'extra_trees': True}, R, Rf, seeds) for nm, (X, Xt, p, R, Rf, seeds) in SPEC1.items()}
    return SPEC1, SPEC1_ET

rk = lambda v: (rankdata(v) - 1.0) / (len(v) - 1.0)
BASE_LGB = dict(objective='binary', force_col_wise=True, bagging_freq=1, verbose=-1, num_threads=os.cpu_count() or 4, seed=42)
W = {'qf_shallow': 0.24, 'qf_reg': 0.12, 'v1_allfam_oh': 0.32, 'v2_allfam_oh': 0.16, 'v2_mono': 0.16}

def run_member(name, spec, tag, view=None):
    X, Xt, p, R, Rf, seeds = spec[name]
    if view is not None and 'monotone_constraints' in p:
        p = {**p, 'monotone_constraints': list(p['monotone_constraints']) + [0] * view['tr'][0].shape[1]}
    tf = np.zeros(nte); oof = np.zeros(ntr)
    ex = lambda key, k=None: ([] if view is None else [view[key] if k is None else view[key][k]])
    for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(ntr), y)):
        fa = pd.concat([X.iloc[a].reset_index(drop=True), train_enc(a)] + ex('tr', k), axis=1)
        fb = pd.concat([X.iloc[b].reset_index(drop=True), enc(a, b)] + ex('va', k), axis=1)
        ft = pd.concat([Xt.reset_index(drop=True), enc(a, tidx)] + ex('te', k), axis=1)
        m = lgb.train({**BASE_LGB, **p}, lgb.Dataset(fa, y[a]), R)
        oof[b] = m.predict(fb); tf += m.predict(ft) / 5.0
    allr = np.arange(ntr)
    fa = pd.concat([X.reset_index(drop=True), train_enc(allr)] + ex('all_tr'), axis=1)
    ft = pd.concat([Xt.reset_index(drop=True), enc(allr, tidx)] + ex('all_te'), axis=1)
    fr = [rk(lgb.train({**BASE_LGB, **p, 'seed': sd}, lgb.Dataset(fa, y), Rf).predict(ft)) for sd in seeds]
    test = rk(0.5 * rk(tf) + 0.5 * np.mean(fr, 0))
    print(f'  [{tag}_{name}] OOF AUC: {roc_auc_score(y, oof):.5f}', flush=True)
    return test, oof

In [ ]:
# Cell 7: the single v17 stream (15 LightGBM members + CatBoost product specialist)
t0 = time.time()
SPEC1_v17, SPEC1_ET_v17 = get_specs()
RESV = {}
for nm in W: RESV[('e1', nm)]    = run_member(nm, SPEC1_v17,    'v17_e1',    VIEW_O)
for nm in W: RESV[('e1et', nm)]  = run_member(nm, SPEC1_ET_v17, 'v17_e1et',  VIEW_O)
for nm in W: RESV[('e1Aet', nm)] = run_member(nm, SPEC1_ET_v17, 'v17_e1Aet', VIEW_A)

print('[*] CatBoost product specialist')
cb_numeric = pd.concat([BASEF['Xa'], BASEF['Xat']], ignore_index=True).drop(columns=CATS).reset_index(drop=True)
cb_cat = al[CATS].copy().reset_index(drop=True)
for c in CATS: cb_cat[c] = cb_cat[c].astype('string').fillna('__MISSING__').astype(str)
cb_product = pd.concat([cb_numeric, cb_cat], axis=1)
product_names = []
for k_ in ['Q3', 'Q4', 'Q5', 'P3', 'P4', 'P5', 'PF4'] + [k for k in ('CT1', 'CT2', 'CT3', 'CT1_GP', 'CT1_CR', 'CT1_AN') if k in KS]:
    cb_product['PROD_' + k_] = KS[k_].reset_index(drop=True).astype('string').fillna('__MISSING__').astype(str); product_names.append('PROD_' + k_)
kd = pd.concat([BASEF['Xak']['K_DEC'], BASEF['Xakt']['K_DEC']], ignore_index=True)
cb_product['PROD_K_DEC'] = kd.round().astype('Int64').astype('string').fillna('__MISSING__').astype(str); product_names.append('PROD_K_DEC')
cb_product['PROD_FEE'] = pd.Series(FEE).reset_index(drop=True).astype(str); product_names.append('PROD_FEE')
CB_CATS = CATS + product_names
cb_train, cb_test = cb_product.iloc[:ntr].reset_index(drop=True), cb_product.iloc[ntr:].reset_index(drop=True)

CBP = dict(iterations=900, depth=7, learning_rate=0.05, loss_function='Logloss', eval_metric='AUC', l2_leaf_reg=8, random_seed=42,
           thread_count=os.cpu_count() or 4, verbose=False, allow_writing_files=False)
cb_oof = np.zeros(ntr); cb_tests = []
for fi, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(ntr), y), 1):
    pe_a, pe_b, pe_t = partner_enc(a, a), partner_enc(a, b), partner_enc(a, tidx)
    ca = pd.concat([cb_train.iloc[a].reset_index(drop=True), pe_a], axis=1)
    cbv = pd.concat([cb_train.iloc[b].reset_index(drop=True), pe_b], axis=1)
    cbt = pd.concat([cb_test.reset_index(drop=True), pe_t], axis=1)
    m = CatBoostClassifier(**CBP); m.fit(ca, y[a], cat_features=CB_CATS)
    pv, pt = m.predict_proba(cbv)[:, 1], m.predict_proba(cbt)[:, 1]
    cb_oof[b] = pv; cb_tests.append(pt)
    print(f'  [CatBoost fold {fi}/5] AUC: {roc_auc_score(y[b], pv):.5f}', flush=True)
cb_test = np.mean(cb_tests, 0)

blend = lambda tag, i: sum(W[k] * (RESV[(tag, k)][0] if i == 0 else rk(RESV[(tag, k)][1])) for k in W)
B_ = lambda i: (blend('e1', i) + blend('e1et', i)) / 2.0
champ = lambda i, cb: 0.9 * rk(0.75 * rk(B_(i)) + 0.25 * blend('e1Aet', i)) + 0.1 * rk(cb)
oof_v17, test_v17 = champ(1, cb_oof), champ(0, cb_test)
print(f'>>> v17 single stream: pooled OOF AUC = {roc_auc_score(y, oof_v17):.6f} | elapsed {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 8: write ONE csv + integrity checks
final = (rankdata(test_v17) - 1.0) / (len(test_v17) - 1.0)
sub = pd.DataFrame({'SK_ID_CURR': df_test['SK_ID_CURR'].values, 'TARGET': final})
assert len(sub) == nte and sub['TARGET'].notna().all() and sub['SK_ID_CURR'].is_unique
assert (sub['SK_ID_CURR'].values == sample['SK_ID_CURR'].values).all()
OUT = 'v_single_stream_v17.csv'
sub.to_csv(OUT, index=False)
print(f'[+] wrote {OUT} ({len(sub)} rows, STRICT_INDUCTIVE={STRICT_INDUCTIVE})')

# Optional parity check: point GCI_V17_REFERENCE_CSV at your saved v17 submission (any file with SK_ID_CURR, TARGET)
ref_path = os.environ.get('GCI_V17_REFERENCE_CSV', 'd:/gci/pre info/oct22/GCI_submission_v17.csv')
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path).set_index('SK_ID_CURR')['TARGET']
    ref = ref.reindex(sub['SK_ID_CURR'].values).values
    print(f'[*] Spearman vs reference v17: {spearmanr(final, ref)[0]:.6f}  (about 0.999 or higher means the same model; much lower means this run differs)')
else:
    print('[*] reference CSV not found; set GCI_V17_REFERENCE_CSV to run the parity check')